# Unit 4 — clustering, CNNs, reinforcement learning, LoRA, MLOps

Phases 8, 9, 11, 12 and 15.

This notebook reads committed results from `reports/`. It reproduces nothing and needs no corpus or GPU. `reports/README.md` names the command that regenerates each file. The full syllabus mapping is in `docs/syllabus_map.md`.

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "reports").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CELLS = json.loads((ROOT / "reports" / "master_results.json").read_text(encoding="utf-8"))["cells"]


def results(*stages):
    """The master table's rows for these stages, as a Markdown table."""
    rows = ["| stage | model | metric | value | target | verdict |", "| :-- | :-- | :-- | --: | --: | :-: |"]
    for c in CELLS:
        if c["stage"] in stages and c.get("available", True):
            verdict = "" if c.get("passes") is None else ("pass" if c["passes"] else "**fail**")
            target = "" if c.get("target") is None else c["target"]
            rows.append(f"| {c['stage']} | {c['model']} | {c['metric']} | {c['value']} | {target} | {verdict} |")
    display(Markdown("\n".join(rows)))


def report(name, *keys):
    """Selected top-level fields of one reports/*.json."""
    data = json.loads((ROOT / "reports" / name).read_text(encoding="utf-8"))
    return {k: data[k] for k in keys if k in data}


## Clustering (8)

Elbow and silhouette both choose K = 3. At K = 3 the ARI against the shape labels is 0.0028, so the structure they find is not shape type.

![p8_choose_k.png](../reports/figures/p8_choose_k.png)

`reports/figures/p8_choose_k.png`

![p8_dendrogram.png](../reports/figures/p8_dendrogram.png)

`reports/figures/p8_dendrogram.png`

![p8_style_clusters.png](../reports/figures/p8_style_clusters.png)

`reports/figures/p8_style_clusters.png`

## CNN — filters, feature maps, detection (9)

The from-scratch CNN has 372,183 parameters (the arithmetic is in `docs/cnn_math.md`). The YOLO detector is S2.

In [2]:
results("detect", "ocr")

| stage | model | metric | value | target | verdict |
| :-- | :-- | :-- | --: | --: | :-: |
| detect | YOLO component detector | mAP@0.5 | 0.9107 | 0.8 | pass |
| detect | YOLO component detector | hand-drawn mAP@0.5 | 0.8787 |  |  |
| ocr | trocr_large + learned selection (incumbent) | CER | 0.2006 | 0.15 | **fail** |
| ocr | trocr_large + learned selection (incumbent) | exact match | 0.6656 |  |  |
| ocr | trocr_large + learned selection (incumbent) | WER | 0.264 |  |  |
| ocr | trocr_large, old selection | CER | 0.2133 |  |  |
| ocr | trocr_large, 28 epochs | CER | 0.2222 |  |  |
| ocr | trocr_large, strong augmentation | CER | 0.2173 |  |  |
| ocr | trocr_large, retrained on reselection | CER | 0.2073 |  |  |
| ocr | trocr_fa (fa_bresler specialist) | best dev CER | 0.3902 |  |  |
| ocr | crnn_finetune (9.3.6) | CER | 0.6864 |  |  |
| ocr | crnn_scratch (9.3.6) | CER | 0.8397 |  |  |
| ocr | trocr_zero_shot (9.3.6) | CER | 1.1702 |  |  |

![p9_filters.png](../reports/figures/p9_filters.png)

`reports/figures/p9_filters.png`

![p9_featuremaps.png](../reports/figures/p9_featuremaps.png)

`reports/figures/p9_featuremaps.png`

![p9_gradcam.png](../reports/figures/p9_gradcam.png)

`reports/figures/p9_gradcam.png`

![p9_detection.png](../reports/figures/p9_detection.png)

`reports/figures/p9_detection.png`

## Q-learning traversal (11)

Implemented and evaluated, but not load-bearing: removing RL costs nothing in executability (11.2.10).

![p11_convergence.png](../reports/figures/p11_convergence.png)

`reports/figures/p11_convergence.png`

![p11_qvalues.png](../reports/figures/p11_qvalues.png)

`reports/figures/p11_qvalues.png`

![p11_sarsa.png](../reports/figures/p11_sarsa.png)

`reports/figures/p11_sarsa.png`

![p11_dqn.png](../reports/figures/p11_dqn.png)

`reports/figures/p11_dqn.png`

![p11_ablation.png](../reports/figures/p11_ablation.png)

`reports/figures/p11_ablation.png`

## QLoRA code synthesis (12)

S6 and S7 are measured from the IR to code, not from a photograph.

In [3]:
results("codegen", "assemble")

| stage | model | metric | value | target | verdict |
| :-- | :-- | :-- | --: | --: | :-: |
| assemble | full assembly (val) | median GED | 3.0 | 3.0 | pass |
| assemble | full assembly (test) | median GED | 13.0 | 3.0 | **fail** |
| assemble | full assembly (test) | mean GED | 15.512 |  |  |
| assemble | full assembly (test) | pass share at <= 3 | 0.2593 |  |  |
| assemble | yolov8m-pose arrows (test) | median GED | 13.0 |  |  |
| assemble | yolov8s arrows (test) | median GED | 14.5 |  |  |
| codegen | Qwen2.5-Coder-7B zero-shot | functional pass@1 | 0.0309 |  |  |
| codegen | Qwen2.5-Coder-7B few-shot k=2 | functional pass@1 | 0.3951 |  |  |
| codegen | LoRA r=64 | functional pass@1 | 0.7037 | 0.7 | pass |
| codegen | LoRA r=64 | executes | 0.9877 | 0.85 | pass |
| codegen | LoRA r=64 | syntax | 0.9938 | 0.95 | pass |
| codegen | LoRA r=64 | contract | 0.9938 |  |  |
| codegen | LoRA r=64 | dropped-node rate | 0.1683 |  |  |
| codegen | LoRA r=64 | invented per program | 0.864 |  |  |
| codegen | LoRA r=64 | structural fidelity | 0.8263 |  |  |
| codegen | LoRA r=64 | CodeBLEU | 0.933223 |  |  |
| codegen | LoRA r=64 | exact match | 0.598765 |  |  |
| codegen | LoRA r=64 | edit similarity | 0.919641 |  |  |
| codegen | LoRA r=64 + one repair turn | functional pass@1 | 0.7099 |  |  |
| codegen | 12.1.6 reference emitter | functional pass@1 | 0.784 |  |  |
| codegen | 12.1.6 reference emitter | structural fidelity | 0.8267 |  |  |

![p12_train_loss.png](../reports/figures/p12_train_loss.png)

`reports/figures/p12_train_loss.png`

![p12_lora_sweep.png](../reports/figures/p12_lora_sweep.png)

`reports/figures/p12_lora_sweep.png`

![p12_three_way.png](../reports/figures/p12_three_way.png)

`reports/figures/p12_three_way.png`

## MLOps (15)

In [4]:
report("drift.json", "what", "thresholds", "verdict")

{'what': '15.5 - PSI / KS drift detection, with the controls that make an alarm mean something',
 'thresholds': {'psi_moderate': 0.1,
  'psi_significant': 0.25,
  'min_drifted_features': 3,
  'bins': 10},
 'verdict': {'controls_pass': True,
  'every_shift_detected': True,
  'usable': True}}

![p15_prediction_monitoring.png](../reports/figures/p15_prediction_monitoring.png)

`reports/figures/p15_prediction_monitoring.png`